# AI Explainability



## 1. Set Up Environment

In [ ]:
#imports
import panada as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import pickle , warnings

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette ='flare', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120})

models_dir = '/Users/wadalisam/Documents/GitHub/insider-threat-detector/models/'
figures_dir = '/Users/wadalisam/Documents/GitHub/insider-threat-detector/figures/'
os.makedirs(figures_dir, exist_ok=True)

with open (f'{models_dir}recall_model.pkl', 'rb') as  f: model = pickle.load(f)
with open (f'{models_dir}encoders.pkl', 'rb') as f: encoders = pickle.load(f)
with open (f'{models_dir}feature_names.pkl', 'rb') as f: feature_names = pickle.load(f)
with open (f'{models_dir}splits.pkl', 'rb') as f: splits = pickle.load(f)

X_test = splits['X_test']
y_test = splits['y_test']

X_test_df = pd.DataFrame(X_test, columns=feature_names)
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:,1]

print("-------------------------------Artifacts Loaded-----------------------------------")

## 2. Feature Importance

In [ ]:
def get_feature_importances(model, feature_names):
    """Extract feature importances from any model or VotingClassifier."""
    if hasattr(model, 'feature_importances_'):
        return pd.Series(model.feature_importances_, index=feature_names)
    
    elif hasattr(model, 'estimators_'):
        # VotingClassifier: .estimators_ is a list of fitted objects (no names)
        # Use .estimators (unfitted, with names) to pair them up
        importances = []
        for estimator in model.estimators_:
            # Unwrap Pipeline (e.g. lr_pipe)
            inner = estimator.named_steps['lr'] if hasattr(estimator, 'named_steps') else estimator
            if hasattr(inner, 'feature_importances_'):
                importances.append(inner.feature_importances_)
        
        if importances:
            avg = np.mean(importances, axis=0)
            return pd.Series(avg, index=feature_names)
        else:
            print(f'No tree-based estimators found in VotingClassifier — cannot compute feature importances.')
            return None
    else:
        print(f'{type(model).__name__} does not support feature importances.')
        return None

In [ ]:
features = get_feature_importances(model, feature_names)

if features is not None:
    fi = features.sort_values(ascending=False)
    print('---------------------Gini Impurity Reduction:--------------------------------------------')
    for feat, score in fi.items():
        bar = '█' * int(score * 200)
        print(f'  {feat:<35} {score:.4f}  {bar}')
flare = sns.color_palette('flare', as_cmap=False, n_colors=10)

# Pick two distinct shades from flare
high_risk_color      = flare[9]   # darkest — primary risk
supporting_color     = flare[3]   # lighter — supporting/profile

high_risk = {'total_files_burned', 'burned_from_other', 'num_printed_pages_off_hours'}
colors_fi = [high_risk_color if f in high_risk else supporting_color for f in fi_sorted.index]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(fi_sorted.index, fi_sorted.values, color=colors_fi, edgecolor='white')
ax.set_xlabel('Feature Importance (Gini)', fontsize=11)
ax.set_title('Random Forest — Global Feature Importance\nfor Insider Threat Detection', fontsize=13, fontweight='bold')

for bar, val in zip(bars, fi_sorted.values):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=8)

red_p  = mpatches.Patch(color=high_risk_color,  label='Primary risk indicator')
blue_p = mpatches.Patch(color=supporting_color, label='Supporting/profile feature')
ax.legend(handles=[red_p, blue_p], loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
for i, (feat, score) in enumerate(fi.head(5).items(), 1):
    print(f'  {i}. {feat}: {score:.4f} ({score*100:.1f}% of total importance)')

## 3. Risk Score Breakdown

In [ ]:
def get_risk_breakdown(row: pd.Series, feature_names: list, model) -> pd.Series:
    """
    Approximates per-feature contribution to the malicious class prediction
    by weighting the feature importance by how extreme its value is
    relative to the test set range.
    """
    # fi is already computed globally — reuse it
    importances = fi  # pd.Series indexed by feature_names

    contributions = {}
    for feat in feature_names:
        val = row[feat]
        feat_range = X_test_df[feat].max() - X_test_df[feat].min()
        normalised = val / feat_range if feat_range > 0 else 0
        contributions[feat] = normalised * importances[feat]

    contrib_series = pd.Series(contributions)
    total = contrib_series.sum()
    return (contrib_series / total * 100).sort_values(ascending=False) if total > 0 else contrib_series

In [ ]:
def explain_prediction(idx: int):
    """Full explanation for a single prediction."""
    row   = X_test_df.iloc[idx]
    pred  = y_pred[idx]
    prob  = y_prob[idx]
    truth = y_test.iloc[idx]

    outcome = ('True Positive'  if pred==1 and truth==1 else
               'True Negative'  if pred==0 and truth==0 else
               'False Positive' if pred==1 and truth==0 else
               'False Negative')

    print(f'-----------------Record #{idx} ---------------------------------------')
    print(f'Ground truth:  {"MALICIOUS" if truth==1 else "NORMAL"}')
    print(f'Prediction:    {"MALICIOUS" if pred==1  else "NORMAL"}')
    print(f'Confidence:    {prob*100:.1f}% malicious')
    print(f'Outcome:       {outcome}')

    breakdown = get_risk_breakdown(row, feature_names, model)
    print('\nTop contributing features:')
    for feat, pct in breakdown.head(6).items():
        val = row[feat]
        print(f'  {feat:<35} value={val:.1f}  contribution≈{pct:.1f}%')
    print()

In [ ]:
tp_idx = next(i for i in range(len(y_pred)) if y_pred[i]==1 and y_test.iloc[i]==1 and y_prob[i]>0.8)
tn_idx = next(i for i in range(len(y_pred)) if y_pred[i]==0 and y_test.iloc[i]==0 and y_prob[i]<0.1)

fp_candidates = [i for i in range(len(y_pred)) if y_pred[i]==1 and y_test.iloc[i]==0]
fn_candidates = [i for i in range(len(y_pred)) if y_pred[i]==0 and y_test.iloc[i]==1]

print(f'Found {len(fp_candidates):,} false positives and {len(fn_candidates):,} false negatives in test set.\n')

for idx in [tp_idx, tn_idx, fp_candidates[0], fn_candidates[0]]:
    explain_prediction(idx)

## 4. Visual Explanation Per Record

In [ ]:
def plot_risk_breakdown(idx: int, ax):
    row       = X_test_df.iloc[idx]
    pred      = y_pred[idx]
    prob      = y_prob[idx]
    truth     = y_test.iloc[idx]
    breakdown = get_risk_breakdown(row, feature_names, model).head(8)

    flare = sns.color_palette('flare', n_colors=10)
    color = flare[9] if pred == 1 else flare[3]   # dark = malicious, light = normal
    ax.barh(breakdown.index, breakdown.values, color=color, edgecolor='white')
    ax.set_xlabel('Contribution (%)')
    outcome = ('TP' if pred==1 and truth==1 else 'TN' if pred==0 and truth==0
               else 'FP' if pred==1 else 'FN')
    ax.set_title(
        f'Record #{idx} — {"MALICIOUS" if pred==1 else "NORMAL"} ({prob*100:.0f}% conf) [{outcome}]',
        fontweight='bold', fontsize=9
    )

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
scenarios = [
    (tp_idx,           'True Positive'),
    (tn_idx,           'True Negative'),
    (fp_candidates[0], 'False Positive'),
    (fn_candidates[0], 'False Negative'),
]
for ax, (idx, label) in zip(axes.flatten(), scenarios):
    plot_risk_breakdown(idx, ax)
    ax.invert_yaxis()

plt.suptitle('Per-Record Feature Contribution — Testing Scenarios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{figures_dir}per_record_breakdown.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Human-Readable Risk Narrative Generator

In [ ]:
def generate_narrative(idx):

    # Get the selected record
    row = X_test_df.iloc[idx]

    # Model outputs
    pred = y_pred[idx]
    prob = y_prob[idx]

    # Determine risk level
    if prob > 0.80:
        risk_label = "CRITICAL"
    elif prob > 0.60:
        risk_label = "HIGH"
    elif prob > 0.30:
        risk_label = "MEDIUM"
    else:
        risk_label = "LOW"

    # Start building report
    lines = []

    lines.append("-----------------Insider Threat Assessment Report-----------------------------------------")
    lines.append(f"Record ID: #{idx}")

    # Prediction result
    if pred == 1:
        verdict = "MALICIOUS"
    else:
        verdict = "NORMAL"

    lines.append(f"Verdict: {verdict}")
    lines.append(f"Risk Level: {risk_label}")
    lines.append(
        f"Confidence: {prob * 100:.1f}% probability of malicious intent"
    )

    lines.append("")
    lines.append("-------------------Risk Indicators Detected:-------------------------------")

    flags = []

    # Files copied to removable media
    if row["total_files_burned"] > 5:
        flags.append(
            f"HIGH: Employee copied "
            f"{row['total_files_burned']:.0f} files to removable media."
        )

    elif row["total_files_burned"] > 0:
        flags.append(
            f"MEDIUM: Employee copied "
            f"{row['total_files_burned']:.0f} files to removable media."
        )

    # Printing outside business hours
    if row["num_printed_pages_off_hours"] > 0:
        flags.append(
            f"HIGH: "
            f"{row['num_printed_pages_off_hours']:.0f} pages were printed "
            f"outside business hours."
        )

    # Copying files from other employees
    if row["burned_from_other"] > 0:
        flags.append(
            f"HIGH: Employee copied "
            f"{row['burned_from_other']:.0f} files belonging to other employees."
        )

    # Weekend building access
    if row["entry_during_weekend"] == 1:
        flags.append(
            "MEDIUM: Employee entered the building during the weekend."
        )

    # Multiple campus visits
    if row["num_unique_campus"] > 1:
        flags.append(
            f"MEDIUM: Employee visited "
            f"{row['num_unique_campus']:.0f} campuses."
        )

    # Hostility country level
    if row["hostility_country_level"] > 0:
        flags.append(
            f"MEDIUM: Hostility country level is "
            f"{row['hostility_country_level']:.0f}/4."
        )

    # Activity abroad
    if row["is_abroad"] == 1:
        flags.append(
            "MEDIUM: Activity occurred while employee was abroad."
        )

    # If no indicators were found
    if len(flags) == 0:
        flags.append(
            "No major risk indicators were detected."
        )

    # Add indicators to report
    lines.extend(flags)

    lines.append("")
    lines.append("RECOMMENDED ACTION:")

    # Determine recommendation
    if pred == 1 and prob > 0.80:
        lines.append(
            "CRITICAL: Escalate immediately to the security team."
        )

    elif pred == 1:
        lines.append(
            "HIGH: Increase monitoring and review employee activity."
        )

    elif prob > 0.30:
        lines.append(
            "MEDIUM: Continue routine monitoring."
        )

    else:
        lines.append(
            "LOW: No action required."
        )

    # Convert list into final report
    report = "\n".join(lines)

    return report

In [ ]:
print(generate_narrative(tp_idx))
print()
print(generate_narrative(fp_candidates[0]))

## 6. Testing Scenarios

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== Test Set Results ===')
print(f'True  Positives (TP) — correctly flagged malicious:         {tp:,}')
print(f'True  Negatives (TN) — correctly cleared normal:            {tn:,}')
print(f'False Positives (FP) — normal employees wrongly flagged:    {fp:,}')
print(f'False Negatives (FN) — malicious employees missed:          {fn:,}')
print()
print(f'Precision: {tp/(tp+fp)*100:.2f}% — of all flagged, this many were actually malicious')
print(f'Recall:    {tp/(tp+fn)*100:.2f}% — of all real threats, this many were detected')
print(f'F1-Score:  {2*tp/(2*tp+fp+fn)*100:.2f}%')

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Normal', 'Malicious']).plot(
    ax=ax, colorbar=False, cmap='flare')
ax.set_title('Confusion Matrix — Recall Model on Test Set', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{figures_dir}confusion_matrix_recall_model.png', dpi=150, bbox_inches='tight')
plt.show()

## False Positive Analysis

In [ ]:
fp_rows = X_test_df[(y_pred == 1) & (y_test.values == 0)].copy()
fn_rows = X_test_df[(y_pred == 0) & (y_test.values == 1)].copy()
tp_rows = X_test_df[(y_pred == 1) & (y_test.values == 1)].copy()

compare = pd.DataFrame({
    'True Positives (mean)':  tp_rows.mean(),
    'False Positives (mean)': fp_rows.mean(),
    'False Negatives (mean)': fn_rows.mean(),
}).T

print('Mean feature values by prediction outcome:')
print(compare[['total_files_burned','num_printed_pages_off_hours',
               'burned_from_other','entry_during_weekend',
               'num_unique_campus','hostility_country_level']].round(3).to_string())

In [ ]:
key_feats = ['total_files_burned', 'num_printed_pages_off_hours', 'burned_from_other',
             'entry_during_weekend', 'num_unique_campus']

flare = sns.color_palette('flare', n_colors=10)
bar_colors = [flare[8], flare[4], flare[1]]   # TP dark, FP mid, FN light

fig, axes = plt.subplots(1, len(key_feats), figsize=(18, 4))
for ax, feat in zip(axes, key_feats):
    vals = [tp_rows[feat].mean(), fp_rows[feat].mean(), fn_rows[feat].mean()]
    ax.bar(['True Pos', 'False Pos', 'False Neg'], vals,
           color=bar_colors, edgecolor='white')
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold', fontsize=8)
    ax.set_ylabel('Mean')

plt.suptitle('Mean Feature Values by Prediction Outcome Group', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{figures_dir}fp_fn_analysis.png', dpi=150, bbox_inches='tight')
plt.show()